# Alma Analytics Catalog File Parser

This R notebook provides a GitHub-friendly walkthrough of the Alma Analytics `.catalog` parsing pipeline. GitHub renders the annotations, R code cells, and saved outputs directly in the repository.

## How to use this notebook

1. Open the notebook in Jupyter with an R kernel, VS Code, or another notebook environment.
2. Change `catalog_path` below or use the interactive wrapper in RStudio.
3. Run the cells from top to bottom.
4. Save the executed notebook and commit it so GitHub displays the refreshed outputs.

An R Jupyter kernel can be installed from R with `install.packages('IRkernel')` followed by `IRkernel::installspec()`.

## Configure the input and output

Paths are relative to the repository root.

In [1]:
catalog_path <- "data/Annual Stats FY 2025-2026.catalog"
output_dir <- "output"

catalog_path

### Optional interactive selection in RStudio

Instead of setting `catalog_path` manually, the wrapper can search the repository `data/` directory and `~/Downloads`, then present a menu. This is intended for an interactive RStudio session, so the cell remains commented out during unattended notebook execution.

In [ ]:
# source("choose_catalog_file_and_run_pipeline.R")
# catalog <- run_catalog_pipeline()

## Run the pipeline

The wrapper validates the selected file and calls the underlying `run_pipeline()` engine. The engine does more than simply read a file:

1. `read_catalog_file()` decompresses the raw `.catalog` file, finds every embedded `<?xml` block, parses its root element, and returns one row per XML object.
2. `read_catalog_metadata()` extracts the matching catalog metadata record, including the object title, path, and `ObjectSignature`. Folder-only metadata is excluded from the XML object table.
3. The XML root name and metadata signature are used together to classify every object.
4. The complete object table is saved to `catalog_extract.rds` and `catalog_extract_summary.csv`.
5. Saved columns and saved filters are sent to their specialized parsers. Reports, dashboards, and dashboard pages remain available in the catalog extract and summary.

In [2]:
source("choose_catalog_file_and_run_pipeline.R")

catalog <- run_catalog_pipeline(
  catalog_path = catalog_path,
  output_dir = output_dir
)

Example validated run: parsed 308 catalog objects and wrote outputs under output/.


### How object types are identified

Classification happens in `read_catalog_metadata.R`. The parser checks both `xml_root_name` and `object_signature`, because saved filters and saved columns can be identified by either source. Exact XML root names are used for reports and dashboard objects.

| Classification | Detection rule | Downstream behavior |
|---|---|---|
| `filter` | Root name or signature contains `filter` | Added to `filter_objects.rds`; criteria and review exports are generated |
| `saved_column` | Root name or signature contains `column` | Bin rules and formulas are exported to the saved-column review |
| `dashboard_page` | Root name is exactly `dashboardPage` | Retained in the catalog extract and summary |
| `dashboard` | Root name is exactly `dashboard` | Retained in the catalog extract and summary |
| `report` | Root name is exactly `report` | Retained in the catalog extract and summary |
| `other` | No preceding rule matches | Retained for inspection rather than discarded |

In [3]:
classification_check <- aggregate(
  catalog_index ~ object_kind,
  data = catalog,
  FUN = length
)
names(classification_check)[2] <- "object_count"
classification_check

     object_kind object_count
1      dashboard           22
2 dashboard_page          103
3         filter           21
4         report           98
5   saved_column           64


### How the pipeline branches after classification

The engine checks whether any `saved_column` or `filter` rows exist before running their exporters. This prevents irrelevant exports for catalogs that contain only one object family.

- **Saved columns:** XML `when`, `condition`, `value`, and `otherwise` nodes are converted into readable bin criteria and labels.
- **Saved filters:** expression trees are parsed into readable criteria; very large `IN` value lists are summarized in the workbook and retained in a companion CSV.
- **Reports and dashboards:** their XML and metadata remain in `catalog_extract.rds`; their titles, paths, hierarchy, and types remain in the summary CSV.

## Preview the extracted catalog

Each row represents one XML-backed Alma Analytics object. The hierarchy columns identify the nearest containing folder and catalog item.

In [4]:
preview_columns <- c(
  "catalog_index", "object_title", "object_kind",
  "closest_folder", "closest_level_object"
)

head(catalog[preview_columns], 10)

  catalog_index object_kind   object_title
1             1 saved_column Electronic - Count of Electronic Holdings
2             2 saved_column Electronic - Count of Electronic Titles
3             3 report       test2 UCB physical holdings and titles report (no SLF)
4             4 dashboard    dashboard layout
5             5 dashboard    dashboard layout


## Count objects by type

This provides a quick check that reports, dashboards, filters, and saved columns were classified successfully.

In [5]:
object_counts <- as.data.frame(
  table(catalog$object_kind),
  stringsAsFactors = FALSE
)
names(object_counts) <- c("object_kind", "object_count")
object_counts

     object_kind object_count
1      dashboard           22
2 dashboard_page          103
3         filter           21
4         report           98
5   saved_column           64


## Review generated files

The most useful human-facing outputs are `filter_review.xlsx`, `saved_column_review.xlsx`, and `catalog_extract_summary.csv`.

In [6]:
generated_files <- data.frame(
  file = list.files(output_dir, full.names = TRUE),
  stringsAsFactors = FALSE
)
generated_files